Ma a modell eszközöket kap: Python-függvényeket, amelyeket meghívhat. A modell csak **kéri**, a mi kódunk **futtatja**. Net kell (árfolyam-API). Az Ollama fusson (`gemma4:e4b`). Futtasd sorban: **Shift + Enter**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import csv
import io
import json
import re
import time
import urllib.request
from datetime import date, timedelta

import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

Segédfüggvény az ellenőrzésekhez (az 5. alkalomból):

In [ ]:
def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    szam = str(szam).replace(".", ",")
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None

## 1. Miért kell eszköz?

Adat: `data/inflacio_eves.csv` (leírása: `data/inflacio_eves.md`).

### Feladat 1.1 – olvasás

Az éves HICP-változások 2021–2024: 5,2; 15,3; 17,0; 3,7 százalék. Összesen hány százalékkal nőtt az árszínvonal 2020 és 2024 között? Írd be a legközelebbit: `41`, `44`, `47` vagy `50`.

In [ ]:
tipp_1_1 = None  # a None helyére írd a tipped

In [ ]:
inflacio_tabla = pd.read_csv("data/inflacio_eves.csv", sep=";")
index = inflacio_tabla.set_index("ev")["eurostat_hicp_index"]
regi, uj = float(index[2020]), float(index[2024])
helyes = (uj / regi - 1) * 100
print(f"HICP-index: 2020: {regi}, 2024: {uj} (2025 = 100)")
print(f"A változás: {helyes:.2f}%. Az éves változások összege: {inflacio_tabla.set_index('ev').loc[2021:2024, 'eurostat_hicp'].sum():.1f}%.")
ellenoriz("tipp_1_1", vart=min([41, 44, 47, 50], key=lambda x: abs(x - helyes)))

Most a modell, fejből: gondolkodás nélkül, aztán gondolkodással (**kb. fél perc**).

In [ ]:
def magyar(x):
    """Szám magyar tizedesvesszővel."""
    return str(x).replace(".", ",")


KERDES_INDEX = (f"A magyar HICP éves átlagos indexe (2025 = 100) 2020-ban {magyar(regi)}, 2024-ben {magyar(uj)} volt. "
                "Hány százalékkal nőtt az árszínvonal 2020 és 2024 között?")

fejbol = kerdez(KERDES_INDEX + " Csak a számot írd le, két tizedesre.", gondolkodas="none")
fejbol_gondolkodva = kerdez(KERDES_INDEX + " Csak a számot írd le, két tizedesre.")
print("Gondolkodás nélkül:", fejbol)
print("Gondolkodással:   ", fejbol_gondolkodva)
print(f"A Python szerint:  {helyes:.2f}")

Mit mondott a modell, és mit a Python? (A 2. alkalommal: 9,63 a 10,63 helyett.)

## 2. Az eszköz: egy Python-függvény és a leírása

### Feladat 2.1 – hibakeresés

A Copilot számológépe. Futtasd, és vesd össze a fenti helyes értékkel:

In [ ]:
def szazalekos_valtozas_v1(regi, uj):
    """Hány százalékkal változott az érték a régihez képest, két tizedesre."""
    return round((uj - regi) / uj * 100, 2)


print(szazalekos_valtozas_v1(regi, uj))

Javítsd ki: írd meg a `szazalekos_valtozas(regi, uj)` függvényt (két tizedesre kerekít).

In [ ]:
# Ide kerül a javítás

In [ ]:
ellenoriz_fuggveny("szazalekos_valtozas", [
    ((regi, uj), round(helyes, 2)),
    ((100, 103.7), 3.7),
    ((382.1, 412.9), round((412.9 / 382.1 - 1) * 100, 2)),
], tures=0.001)   # szigorú tűrés: a két tizedesre kerekítés is a feladat része

Az én változatom (ezzel megy tovább a notebook):

In [ ]:
def szazalekos_valtozas(regi, uj):
    """Hány százalékkal változott az érték a régihez képest, két tizedesre."""
    return round((uj - regi) / regi * 100, 2)

### Az eszközleírás

A modell nem a kódot látja, hanem egy JSON-leírást: név, mit csinál, paraméterek.

In [ ]:
ESZKOZ_VALTOZAS = {
    "type": "function",
    "function": {
        "name": "szazalekos_valtozas",
        "description": "Kiszámolja, hány százalékkal nőtt (vagy csökkent) egy érték, például egy árindex "
                       "vagy egy árfolyam, a régi értékhez képest. Két tizedesre kerekít.",
        "parameters": {
            "type": "object",
            "properties": {
                "regi": {"type": "number", "description": "a kiinduló érték"},
                "uj": {"type": "number", "description": "az új érték"},
            },
            "required": ["regi", "uj"],
        },
    },
}

### Feladat 2.2 – olvasás

Mit kap meg a modell a `szazalekos_valtozas`-ból?

- `"a"`: a függvény kódját;
- `"b"`: csak a nevét, a leírását és a paraméterei leírását;
- `"c"`: a függvény eredményét minden lehetséges bemenetre.

A leírás tokenekbe kerül: a második cella megméri, mennyibe (egy egytokenes hívás, mint az 5. alkalommal).

In [ ]:
tipp_2_2 = None  # "a", "b" vagy "c"

In [ ]:
ellenoriz("tipp_2_2", vart="b")

proba = [{"role": "user", "content": KERDES_INDEX}]
chat(proba, gondolkodas="none", max_tokens=1)
eszkoz_nelkul = utolso_hasznalat["prompt_tokens"]
chat(proba, eszkozok=[ESZKOZ_VALTOZAS], gondolkodas="none", max_tokens=1)
print(f"A kérés eszköz nélkül {eszkoz_nelkul} token, a leírással {utolso_hasznalat['prompt_tokens']} token.")

## 3. A modell csak kér

A kérdés mellé az eszköz leírása. Mi van a válasz `content` és `tool_calls` mezőjében?

In [ ]:
RENDSZER = ("Magyarul válaszolj, röviden. Ha számolni kell, vagy adat kell, használd az eszközöket, "
            "és számot csak az eszközök eredményéből írj.")
uzenetek = [{"role": "system", "content": RENDSZER}, {"role": "user", "content": KERDES_INDEX}]

valasz = chat(uzenetek, eszkozok=[ESZKOZ_VALTOZAS], gondolkodas="none")
print(json.dumps(valasz, indent=2, ensure_ascii=False))

A modell nem válaszolt, hanem **kért**. Az ellenőrzés a `tool_calls`-t nézi, nem a szöveget:

In [ ]:
def kert_valtozast(v):
    """A modell a szazalekos_valtozas eszközt kérte, a táblázat két számával."""
    hivasok = v.get("tool_calls") or []
    if len(hivasok) != 1 or hivasok[0]["function"]["name"] != "szazalekos_valtozas":
        return False
    arg = json.loads(hivasok[0]["function"]["arguments"])
    return arg.get("regi") == regi and arg.get("uj") == uj


ellenoriz_feltetel("valasz", kert_valtozast, f"a modell a szazalekos_valtozas eszközt kérte, regi={regi}, uj={uj}")

## 4. A kör: mi futtatjuk, a modell válaszol

Kérés kiolvasása → függvény a szótárból → futtatás → eredmény az üzenetlistába (`role: "tool"`) → újra a modell.

In [ ]:
ESZKOZOK = {"szazalekos_valtozas": szazalekos_valtozas}     # név → Python-függvény

hivas = valasz["tool_calls"][0]
nev = hivas["function"]["name"]                               # 1. melyik függvény
argumentumok = json.loads(hivas["function"]["arguments"])     #    milyen argumentumokkal (szövegből szótár)
fuggveny = ESZKOZOK[nev]                                      # 2. kikeresés a szótárból
eredmeny = fuggveny(**argumentumok)                           # 3. futtatás
print(nev, argumentumok, "->", eredmeny)

In [ ]:
uzenetek.append(valasz)                                       # 4. a modell kérése az előzményekbe
uzenetek.append({"role": "tool", "tool_call_id": hivas["id"], "content": json.dumps(eredmeny)})   # és az eredmény

vegso = chat(uzenetek, eszkozok=[ESZKOZ_VALTOZAS], gondolkodas="none")
print(vegso["content"])
print(f"\nAz üzenetlista: {[u['role'] for u in uzenetek]} + a válasz")

In [ ]:
ellenoriz_feltetel("vegso", lambda v: szam_benne(f"{helyes:.2f}", v["content"]) or szam_benne(f"{helyes:.1f}", v["content"]),
                   f"a válaszban a Python eredménye áll ({helyes:.2f}, vagy egy tizedesre kerekítve)")

A modell a mi számunkat fogalmazta meg.

### Feladat 4.1 – Copilottal

> Írj egy `eszkoz_uzenet(hivas)` függvényt. A `hivas` a modell válaszának `tool_calls` listájából egy elem (szótár). A függvény a `hivas["function"]["name"]` alapján kikeresi a függvényt az `ESZKOZOK` szótárból, a `hivas["function"]["arguments"]` JSON-szövegből `json.loads`-szal argumentumokat csinál, lefuttatja vele a függvényt, és ezt az üzenetet adja vissza: `{"role": "tool", "tool_call_id": hivas["id"], "content": json.dumps(eredmeny, ensure_ascii=False)}`. Ha nincs ilyen nevű eszköz az `ESZKOZOK`-ban, az eredmény legyen `{"hiba": "nincs ilyen eszköz: <név>"}`, és ne dobjon hibát.

Olvasd el: mi történik, ha a modell egy nem létező eszközt kér?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
def _jo_uzenet(f):
    if not callable(f):
        return False
    proba = {"id": "proba_1", "type": "function",
             "function": {"name": "szazalekos_valtozas", "arguments": '{"regi": 100, "uj": 103.7}'}}
    nincs = {"id": "proba_2", "type": "function", "function": {"name": "repulj_a_holdra", "arguments": "{}"}}
    try:
        u1, u2 = f(proba), f(nincs)
        return (u1["role"] == "tool" and u1["tool_call_id"] == "proba_1"
                and json.loads(u1["content"]) == szazalekos_valtozas(100, 103.7)
                and u2["tool_call_id"] == "proba_2" and "hiba" in json.loads(u2["content"]))
    except Exception as hiba:   # egy rossz megoldás hibát dobhat: azt is HIBA-ként mutatjuk, a hibaüzenettel
        print(f"A függvény hibát dobott: {type(hiba).__name__}: {hiba}")
        return False


ellenoriz_feltetel("eszkoz_uzenet", _jo_uzenet,
                   "a jó függvény fut, az eredmény role: tool üzenetben jön vissza; ismeretlen eszközre hibaüzenet")

Az én változatom, és két segédfüggvény: `egy_kor` (egy teljes kör) és `hivott_eszkozok` (mit kért a modell):

In [ ]:
def eszkoz_uzenet(hivas, eszkozok=None):
    """Egy eszközhívás végrehajtása: a modell kérése (a tool_calls egy eleme) → a válaszüzenet (role: tool)."""
    eszkozok = eszkozok or ESZKOZOK
    nev = hivas["function"]["name"]
    if nev in eszkozok:
        argumentumok = json.loads(hivas["function"]["arguments"] or "{}")   # a modell szövegként küldi
        fuggveny = eszkozok[nev]                                          # a szótárból
        eredmeny = fuggveny(**argumentumok)
    else:
        eredmeny = {"hiba": f"nincs ilyen eszköz: {nev}"}
    return {"role": "tool", "tool_call_id": hivas["id"], "content": json.dumps(eredmeny, ensure_ascii=False)}

In [ ]:
RENDSZER = ("Magyarul válaszolj, röviden. Ha számolni kell, vagy adat kell, használd az eszközöket, "
            "és számot csak az eszközök eredményéből írj.")


def hivott_eszkozok(uzenetek):
    """Az üzenetlistában a modell által kért eszközhívások: (név, argumentumok) párok, sorrendben."""
    hivasok = []
    for u in uzenetek:
        for h in u.get("tool_calls") or []:
            hivasok.append((h["function"]["name"], json.loads(h["function"]["arguments"] or "{}")))
    return hivasok


def egy_kor(kerdes, leirasok, eszkozok=None):
    """Egy kör: kérdés → a modell eszközt kér → mi futtatjuk → a modell válaszol. Az üzenetlistát adja vissza."""
    uzenetek = [{"role": "system", "content": RENDSZER}, {"role": "user", "content": kerdes}]
    valasz = chat(uzenetek, eszkozok=leirasok, gondolkodas="none")
    uzenetek.append(valasz)
    for hivas in valasz.get("tool_calls") or []:
        uzenet = eszkoz_uzenet(hivas, eszkozok)
        print(f"ESZKÖZ  {hivas['function']['name']}({hivas['function']['arguments']}) -> {uzenet['content']}")
        uzenetek.append(uzenet)
    if valasz.get("tool_calls"):
        uzenetek.append(chat(uzenetek, eszkozok=leirasok, gondolkodas="none"))
    else:
        print("A modell nem kért eszközt.")
    print("VÁLASZ ", uzenetek[-1]["content"])
    return uzenetek

### Ha az eszköz hibás

Ugyanaz a kör, a 2.1-es hibás számológéppel:

In [ ]:
uzenetek_hibas = egy_kor(KERDES_INDEX, [ESZKOZ_VALTOZAS], {"szazalekos_valtozas": szazalekos_valtozas_v1})

Észrevette a modell, hogy az eszköz rossz? Ki ellenőrzi az eszközt?

## 5. Amit a modell nem tudhat: a mai dátum

Eszköz nélkül:

In [ ]:
print(kerdez("Milyen nap van ma? Csak a dátumot írd le.", gondolkodas="none"))

Egy paraméter nélküli eszköz; a modell két eszköz közül választ:

In [ ]:
def mai_datum():
    """A mai dátum ÉÉÉÉ-HH-NN formában."""
    return date.today().isoformat()


ESZKOZ_DATUM = {
    "type": "function",
    "function": {
        "name": "mai_datum",
        "description": "A mai dátum, ÉÉÉÉ-HH-NN formában. Akkor használd, ha a válaszhoz tudni kell, milyen nap van ma.",
        "parameters": {"type": "object", "properties": {}},
    },
}

In [ ]:
ESZKOZOK["mai_datum"] = mai_datum
uzenetek_datum = egy_kor("Milyen nap van ma?", [ESZKOZ_VALTOZAS, ESZKOZ_DATUM])

In [ ]:
ma = date.today()
honapok = ["január", "február", "március", "április", "május", "június", "július", "augusztus",
           "szeptember", "október", "november", "december"]
ma_szovegben = [ma.isoformat(), f"{ma.year}. {honapok[ma.month - 1]} {ma.day}", f"{ma.year}.{ma.month:02d}.{ma.day:02d}"]

ellenoriz_feltetel("uzenetek_datum", lambda u: [n for n, _ in hivott_eszkozok(u)] == ["mai_datum"],
                   "a modell a mai_datum eszközt kérte (a tool_calls alapján), mást nem")
ellenoriz_feltetel("uzenetek_datum", lambda u: any(s in u[-1]["content"] for s in ma_szovegben),
                   f"a válaszban a mai dátum áll ({ma.isoformat()})")

Miért kell két ellenőrzés, és miért nem elég a szöveg?

### Feladat 5.1 – olvasás

Ugyanez a két eszköz, és a kérdés: „Mi a HICP? Egy mondatban.” Hány eszközhívást kér a modell? `0`, `1` vagy `2`?

In [ ]:
tipp_5_1 = None  # a None helyére írd a tipped

In [ ]:
uzenetek_hicp = egy_kor("Mi a HICP? Egy mondatban.", [ESZKOZ_VALTOZAS, ESZKOZ_DATUM])
ellenoriz("tipp_5_1", vart=len(hivott_eszkozok(uzenetek_hicp)))

Ki döntötte el, hogy kell-e eszköz?

## 6. Adat a netről: az euró árfolyama

A Frankfurter API az EKB referenciaárfolyamait adja, kulcs nélkül. Példa: <https://api.frankfurter.dev/v1/2024-01-02?base=EUR&symbols=HUF>

### Feladat 6.1 – Copilottal

> Írj egy `arfolyam(datum)` függvényt, amely a `https://api.frankfurter.dev/v1/{datum}?base=EUR&symbols=HUF` címről lekéri az euró forintárfolyamát a megadott napra (`datum`: "ÉÉÉÉ-HH-NN"), a `urllib.request` modullal. A kérésben legyen `User-Agent` fejléc (például `"kurzus-notebook"`), különben a szerver elutasítja. Ezt a szótárat adja vissza: `{"datum": <a válasz date mezője>, "eur_huf": <a válasz rates mezőjéből a HUF>}`. Ne használj külső csomagot.

Olvasd el: honnan veszi a dátumot, a kérésből vagy a válaszból?

In [ ]:
# Ide kerül a Copilot kódja

Az ellenőrzés független forrásból, az EKB saját adatbázisából dolgozik:

In [ ]:
def ekb_arfolyam(datum):
    """Független ellenőrzés: az EKB saját adatbázisából az utolsó árfolyam a megadott napon vagy előtte."""
    nap = date.fromisoformat(datum)
    url = ("https://data-api.ecb.europa.eu/service/data/EXR/D.HUF.EUR.SP00.A"
           f"?startPeriod={nap - timedelta(days=7)}&endPeriod={nap}&format=csvdata")
    keres = urllib.request.Request(url, headers={"User-Agent": "kurzus-notebook"})
    with urllib.request.urlopen(keres, timeout=30) as valasz:
        sorok = list(csv.DictReader(io.StringIO(valasz.read().decode("utf-8"))))
    utolso = sorok[-1]
    return {"datum": utolso["TIME_PERIOD"], "eur_huf": float(utolso["OBS_VALUE"])}

In [ ]:
ellenoriz_fuggveny("arfolyam", [(("2024-01-02",), ekb_arfolyam("2024-01-02")),
                                  (("2024-01-06",), ekb_arfolyam("2024-01-06"))])

Mit adott a szombatra? Az én változatom, a leírása, és a kérdés eszköz nélkül, aztán eszközzel:

In [ ]:
def arfolyam(datum):
    """Az euró EKB-referenciaárfolyama forintban egy napon (ÉÉÉÉ-HH-NN), a Frankfurter API-ból."""
    url = f"https://api.frankfurter.dev/v1/{datum}?base=EUR&symbols=HUF"
    keres = urllib.request.Request(url, headers={"User-Agent": "kurzus-notebook"})
    with urllib.request.urlopen(keres, timeout=20) as valasz:
        adat = json.loads(valasz.read())
    return {"datum": adat["date"], "eur_huf": adat["rates"]["HUF"]}


ESZKOZ_ARFOLYAM = {
    "type": "function",
    "function": {
        "name": "arfolyam",
        "description": "Az euró hivatalos EKB-referenciaárfolyama forintban egy adott napon. Hétvégén és "
                       "ünnepnapon az előző munkanap árfolyamát adja, a valódi dátummal.",
        "parameters": {
            "type": "object",
            "properties": {"datum": {"type": "string", "description": "a nap, ÉÉÉÉ-HH-NN formában"}},
            "required": ["datum"],
        },
    },
}

ESZKOZOK["arfolyam"] = arfolyam

In [ ]:
KERDES_ARFOLYAM = "Mennyi volt az euró hivatalos árfolyama forintban 2024. január 2-án?"
print("Eszköz nélkül:", kerdez(KERDES_ARFOLYAM, gondolkodas="none"))
print()
uzenetek_arfolyam = egy_kor(KERDES_ARFOLYAM, [ESZKOZ_VALTOZAS, ESZKOZ_DATUM, ESZKOZ_ARFOLYAM])

In [ ]:
vart_arfolyam = ekb_arfolyam("2024-01-02")["eur_huf"]
ellenoriz_feltetel("uzenetek_arfolyam", lambda u: ("arfolyam", {"datum": "2024-01-02"}) in hivott_eszkozok(u),
                   "a modell az arfolyam eszközt kérte, a 2024-01-02 dátummal")
ellenoriz_feltetel("uzenetek_arfolyam", lambda u: szam_benne(vart_arfolyam, u[-1]["content"]),
                   f"a válaszban az EKB árfolyama áll ({magyar(vart_arfolyam)})")

Mit mondott eszköz nélkül? Hogyan lett a „2024. január 2-án”-ból függvényhívás?

## 7. Ha egy kör nem elég

> Hány százalékkal változott az euró forintárfolyama 2024. január 2. és 2025. január 2. között?

Két árfolyam, aztán egy számolás: több kör kell. A `lepes` egy lépést tesz. Futtasd a cellákat egymás után: mit kér a modell, és hogyan nő a tokenszám?

In [ ]:
def lepes(uzenetek, leirasok):
    """Egy lépés: modellhívás; ha eszközt kér, lefuttatjuk, és az eredményt visszaírjuk.
    True-t ad, ha a modell már szövegesen válaszolt (kész)."""
    if uzenetek[-1]["role"] == "assistant" and not uzenetek[-1].get("tool_calls"):
        print("A modell már válaszolt, nincs több lépés.")
        return True
    valasz = chat(uzenetek, eszkozok=leirasok, gondolkodas="none")
    uzenetek.append(valasz)
    print(f"(a kérés {utolso_hasznalat['prompt_tokens']} token)")
    if not valasz.get("tool_calls"):
        print("VÁLASZ ", valasz["content"])
        return True
    for hivas in valasz["tool_calls"]:
        uzenet = eszkoz_uzenet(hivas)
        print(f"ESZKÖZ  {hivas['function']['name']}({hivas['function']['arguments']}) -> {uzenet['content']}")
        uzenetek.append(uzenet)
    return False


OSSZES = [ESZKOZ_VALTOZAS, ESZKOZ_DATUM, ESZKOZ_ARFOLYAM]
uzenetek_ket = [{"role": "system", "content": RENDSZER},
                {"role": "user", "content": "Hány százalékkal változott az euró forintárfolyama 2024. január 2. és 2025. január 2. között?"}]
kesz = lepes(uzenetek_ket, OSSZES)        # 1. lépés

In [ ]:
kesz = lepes(uzenetek_ket, OSSZES)        # 2. lépés

In [ ]:
kesz = lepes(uzenetek_ket, OSSZES)        # 3. lépés

In [ ]:
kesz = lepes(uzenetek_ket, OSSZES)        # 4. lépés, ha kell

In [ ]:
a, b = ekb_arfolyam("2024-01-02")["eur_huf"], ekb_arfolyam("2025-01-02")["eur_huf"]
vart_valtozas = round((b / a - 1) * 100, 2)
print(f"Az EKB szerint: {a} → {b}, a változás {vart_valtozas}%")

ellenoriz_feltetel("uzenetek_ket", lambda u: {("arfolyam", (("datum", "2024-01-02"),)), ("arfolyam", (("datum", "2025-01-02"),)),
                                              ("szazalekos_valtozas", (("regi", a), ("uj", b)))}
                   <= {(n, tuple(sorted(arg.items()))) for n, arg in hivott_eszkozok(u)},
                   "a modell lekérte a két árfolyamot, és a szazalekos_valtozas eszközzel számolt")
ellenoriz_feltetel("uzenetek_ket", lambda u: kesz and szam_benne(vart_valtozas, u[-1]["content"]),
                   f"a válaszban az EKB adataiból számolt változás áll ({magyar(vart_valtozas)}%)")

Hány lépés kellett? Mi nőtt lépésről lépésre, és miért? A 9. alkalommal ezt egy `while` ciklus ismétli.

## 8. A projektben: „és mennyi lett?”

A sáv a nyers PDF-ből; a tényadat eszköze a táblából dolgozik, és a forrást (KSH vagy Eurostat) a modell választja.

In [ ]:
forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")
s = forrasok[forrasok["jelentes"] == "2023-09"].iloc[0]
pdf = kurzus.letolt(s["url"], "data/mnb/" + s["fajl"])
szoveg = " ".join(re.sub(r"\s+", " ", oldal.get_text()) for oldal in pymupdf.open(pdf))
also, felso = re.search(r"Az infláció 2023-ban (\d+,\d) – (\d+,\d) százalék között alakulhat", szoveg).groups()
print(f"A 2023. szeptemberi jelentés: {also}–{felso} százalék")

In [ ]:
def inflacio(ev, forras):
    """Magyarország éves átlagos inflációja (%) egy évben, a KSH vagy az Eurostat adataiból."""
    oszlop = {"KSH": "ksh_fogyasztoiar", "Eurostat": "eurostat_hicp"}[forras]
    sor = inflacio_tabla[inflacio_tabla["ev"] == int(ev)]
    if sor.empty:
        return {"hiba": f"nincs adat erre az évre: {ev}"}
    return {"ev": int(ev), "forras": forras, "valtozas_szazalek": float(sor[oszlop].iloc[0])}


ESZKOZ_INFLACIO = {
    "type": "function",
    "function": {
        "name": "inflacio",
        "description": "Magyarország éves átlagos inflációja (%) egy adott évben (2015-2025), a KSH "
                       "fogyasztóiár-indexéből vagy az Eurostat harmonizált indexéből (HICP).",
        "parameters": {
            "type": "object",
            "properties": {
                "ev": {"type": "integer", "description": "az év, például 2023"},
                "forras": {"type": "string", "enum": ["KSH", "Eurostat"], "description": "az adatforrás"},
            },
            "required": ["ev", "forras"],
        },
    },
}

ESZKOZOK["inflacio"] = inflacio

In [ ]:
ELOZMENY = f"Az MNB 2023. szeptemberi Inflációs jelentése 2023-ra {also}–{felso} százalékos éves átlagos inflációt várt."

uzenetek_projekt_1 = egy_kor(ELOZMENY + " Mennyi lett, és a sávon belül van?", [ESZKOZ_VALTOZAS, ESZKOZ_INFLACIO])
print("\nA modell ezeket az eszközöket kérte:", hivott_eszkozok(uzenetek_projekt_1))

Kért eszközt a modell? Honnan van a válaszban a 17,6? Pontosítsuk a kérdést:

In [ ]:
uzenetek_projekt = egy_kor(ELOZMENY + " Mennyi lett a tényleges éves infláció 2023-ban, és a sávon belül van?",
                           [ESZKOZ_VALTOZAS, ESZKOZ_INFLACIO])

In [ ]:
ellenoriz_feltetel("uzenetek_projekt", lambda u: ("inflacio", {"ev": 2023, "forras": "KSH"}) in hivott_eszkozok(u),
                   "a modell a KSH-adatot kérte le 2023-ra (az MNB a fogyasztóiár-indexet jelzi előre)")

teny = inflacio(2023, "KSH")["valtozas_szazalek"]
a_savban = float(also.replace(",", ".")) <= teny <= float(felso.replace(",", "."))
print(f"A Python szerint: {teny}% {'a sávon belül' if a_savban else 'a sávon kívül'} ({also}–{felso}).")

Jó a forrás? Jó a szám? Jó a következtetés (belül van-e a sávon)? Mit kell kódra bízni?

### Otthonra

A 7. alkalom `agens_terv.md`-jéből (párban: közösen):

1. a notebook végén mindenhol `RENDBEN` legyen (a 6.1-hez net kell);
2. a projekted adatlekérő függvényéhez (a csomag `adatlekeres.py`-jában) írj **eszközleírást** JSON-ben, a `ESZKOZ_INFLACIO` mintájára: név, leírás (mit ad, milyen mértékegységben, mikor használja a modell), paraméterek típussal és leírással. Tedd az `eszkozok.py`-ba, a függvény mellé;
3. hívd meg a modellel egy körben (`egy_kor`) a projekted egyik adatos tesztkérdésére, és ellenőrizd a `hivott_eszkozok`-kal, hogy a jó eszközt, a jó argumentumokkal kérte-e;
4. ha a modell a szöveges következtetésben hibázik, írd le a `feltetelezesek.md`-be, és gondold végig, melyik lépést kellene kódra bízni.